*This notebook scores synthetic size illusions with several vision models.*

# Evaluate Vision Models on Size Illusions

In [ ]:
# Install Python packages used below.
!pip3 install -q torch torchvision transformers pillow matplotlib pandas


In [ ]:
# Imports, data paths, and pick CPU or GPU.
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image


def resolve_data_root():
    preferred = Path('mnt/cs/cs153/data/anika-livia')
    local_fallback = (Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()) / 'data'
    env_override = os.environ.get('CS153_DATA_ROOT')
    if env_override:
        p = Path(env_override).resolve()
        p.mkdir(parents=True, exist_ok=True)
        return p
    try:
        preferred.mkdir(parents=True, exist_ok=True)
        test = preferred / '.write_test'
        test.write_text('ok', encoding='utf-8')
        test.unlink()
        return preferred
    except Exception:
        local_fallback.mkdir(parents=True, exist_ok=True)
        return local_fallback.resolve()


PROJECT_ROOT = Path(os.environ.get('CS153_PROJECT_ROOT', 'mnt/cs/cs153/projects/size_illusion_project')).resolve()
try:
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
except Exception:
    PROJECT_ROOT = ((Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()) / 'project').resolve()
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

DATA_ROOT = resolve_data_root()

CODE_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from data.generate_illusions import generate_dataset
from datasets.synthetic_loader import SyntheticIllusionDataset, ID_TO_LABEL, LABEL_TO_ID
from models.cnn_model import build_resnet50, build_efficientnet_b0, extract_features, train_probe, predict_probe_task_masked
from models.vit_model import build_vit_b16
from models.clip_eval import build_clip, clip_predict

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATA_ROOT =', DATA_ROOT)
device


In [ ]:
# Create synthetic train and test images plus label files.
data_root = DATA_ROOT / 'synthetic_illusions'
data_root.mkdir(parents=True, exist_ok=True)
train_n = 1800
test_n = 600
generate_dataset(data_root, 'train', train_n, with_context=False, seed=10)
generate_dataset(data_root, 'test', test_n, with_context=True, seed=11)
print('synthetic data ready at', data_root, 'train_n=', train_n, 'test_n=', test_n)


In [ ]:
# Load pretrained ResNet50, EfficientNet-B0, ViT-B/16, and CLIP.
resnet, res_tfm, res_dim = build_resnet50(device)
effnet, eff_tfm, eff_dim = build_efficientnet_b0(device)
vit, vit_tfm, vit_dim = build_vit_b16(device)
clip_model, clip_processor = build_clip(device)

In [ ]:
# Extract features, train linear probes for 80 epochs, and run masked predictions.
probe_epochs = 80

train_res = SyntheticIllusionDataset(data_root, 'train', transform=res_tfm)
test_res = SyntheticIllusionDataset(data_root, 'test', transform=res_tfm)
x_tr_r, y_tr, train_rows = extract_features(resnet, train_res, device)
x_te_r, y_te, rows_te = extract_features(resnet, test_res, device)
head_r = train_probe(x_tr_r, y_tr, res_dim, epochs=probe_epochs, device=device)
pred_r, _ = predict_probe_task_masked(head_r, x_te_r, rows_te, device=device)
pred_train_r, _ = predict_probe_task_masked(head_r, x_tr_r, train_rows, device=device)

train_eff = SyntheticIllusionDataset(data_root, 'train', transform=eff_tfm)
test_eff = SyntheticIllusionDataset(data_root, 'test', transform=eff_tfm)
x_tr_e, y_tr_e, _ = extract_features(effnet, train_eff, device)
x_te_e, y_te_e, _ = extract_features(effnet, test_eff, device)
head_e = train_probe(x_tr_e, y_tr_e, eff_dim, epochs=probe_epochs, device=device)
pred_e, _ = predict_probe_task_masked(head_e, x_te_e, rows_te, device=device)
pred_train_e, _ = predict_probe_task_masked(head_e, x_tr_e, train_rows, device=device)

train_vit = SyntheticIllusionDataset(data_root, 'train', transform=vit_tfm)
test_vit = SyntheticIllusionDataset(data_root, 'test', transform=vit_tfm)
x_tr_v, y_tr_v, _ = extract_features(vit, train_vit, device)
x_te_v, y_te_v, _ = extract_features(vit, test_vit, device)
head_v = train_probe(x_tr_v, y_tr_v, vit_dim, epochs=probe_epochs, device=device)
pred_v, _ = predict_probe_task_masked(head_v, x_te_v, rows_te, device=device)
pred_train_v, _ = predict_probe_task_masked(head_v, x_tr_v, train_rows, device=device)


In [ ]:
# Define accuracy and mismatch helpers.
def acc_and_bias(pred_ids, gt_ids):
    """Return mean exact match and mean mismatch (1 - accuracy)."""
    n = len(gt_ids)
    acc = (pred_ids == gt_ids).float().mean().item()
    illusion_bias = (pred_ids != gt_ids).float().mean().item()
    return acc, illusion_bias


In [ ]:
# Build the results table and run CLIP on a subset of images.
gt_train = y_tr
no_ill_acc_r, _ = acc_and_bias(pred_train_r, gt_train)
no_ill_acc_e, _ = acc_and_bias(pred_train_e, y_tr_e)
no_ill_acc_v, _ = acc_and_bias(pred_train_v, y_tr_v)

test_gt = y_te
ill_acc_r, ill_bias_r = acc_and_bias(pred_r, test_gt)
ill_acc_e, ill_bias_e = acc_and_bias(pred_e, y_te_e)
ill_acc_v, ill_bias_v = acc_and_bias(pred_v, y_te_v)

clip_train = SyntheticIllusionDataset(data_root, 'train', transform=None)
clip_test = SyntheticIllusionDataset(data_root, 'test', transform=None)
clip_train_max = min(180, len(clip_train))
clip_test_max = min(180, len(clip_test))
clip_pred_train, clip_gt_train = [], []
for i in range(clip_train_max):
    im, y, row = clip_train[i]
    p, _, _ = clip_predict(clip_model, clip_processor, im, row=row, device=device)
    clip_pred_train.append(LABEL_TO_ID[p])
    clip_gt_train.append(y)
clip_pred_test, clip_gt_test = [], []
for i in range(clip_test_max):
    im, y, row = clip_test[i]
    p, _, _ = clip_predict(clip_model, clip_processor, im, row=row, device=device)
    clip_pred_test.append(LABEL_TO_ID[p])
    clip_gt_test.append(y)
clip_pred_train = torch.tensor(clip_pred_train)
clip_gt_train = torch.tensor(clip_gt_train)
clip_pred_test = torch.tensor(clip_pred_test)
clip_gt_test = torch.tensor(clip_gt_test)
no_ill_acc_c, _ = acc_and_bias(clip_pred_train, clip_gt_train)
ill_acc_c, ill_bias_c = acc_and_bias(clip_pred_test, clip_gt_test)

synthetic_results = pd.DataFrame(
    [
        ['ResNet50', no_ill_acc_r, ill_acc_r, ill_bias_r],
        ['EfficientNet-B0', no_ill_acc_e, ill_acc_e, ill_bias_e],
        ['ViT-B/16', no_ill_acc_v, ill_acc_v, ill_bias_v],
        ['CLIP', no_ill_acc_c, ill_acc_c, ill_bias_c],
    ],
    columns=['Model', 'No-illusion acc', 'Illusion acc', 'Illusion bias'],
)
print('CLIP synthetic evaluated on', clip_train_max, 'train and', clip_test_max, 'test samples for CPU speed')
synthetic_results


In [ ]:
# Show example test images with ResNet predictions.
rows = SyntheticIllusionDataset(data_root, "test", transform=None).rows
pred_lbl_res = [ID_TO_LABEL[int(x)] for x in pred_r]
gt_lbl = [r["ground_truth"] for r in rows]
correct_idx = [i for i, (p, g) in enumerate(zip(pred_lbl_res, gt_lbl)) if p == g][:3]
fooled_idx = [i for i, (p, g) in enumerate(zip(pred_lbl_res, gt_lbl)) if p != g][:3]

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for col, idx in enumerate(correct_idx):
    row = rows[idx]
    im = Image.open(data_root / row["image_path"])
    axes[0, col].imshow(im)
    axes[0, col].set_title(f"correct\nGT={row['ground_truth']} Pred={pred_lbl_res[idx]}")
    axes[0, col].axis("off")
for col, idx in enumerate(fooled_idx):
    row = rows[idx]
    im = Image.open(data_root / row["image_path"])
    axes[1, col].imshow(im)
    axes[1, col].set_title(f"fooled\nGT={row['ground_truth']} Pred={pred_lbl_res[idx]}")
    axes[1, col].axis("off")
plt.tight_layout()
plt.show()